# Candlestick Pattern Detection
by Chee-Foong 


## Summary 
This analysis follows the steps found in the article "[Recognizing over 50 Candlestick Patterns with Python](https://medium.com/analytics-vidhya/recognizing-over-50-candlestick-patterns-with-python-4f02a1822cb5)" by **Caner Irfanoglu**.  I thank him for making his work public.

Candlestick patterns is one of many features for machine learning modeling work to predict time-series returns of investible assets like shares, currency, cryptos, etc.


## Reference
1. https://medium.com/analytics-vidhya/recognizing-over-50-candlestick-patterns-with-python-4f02a1822cb5
2. https://github.com/mrjbq7/ta-lib
3. https://en.wikipedia.org/wiki/Candlestick_pattern
4. https://www.youtube.com/watch?v=sJCgnSOcTPE

In [1]:
# !pip3 install TA-Lib

## Initialisation

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=Warning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# IMPORTS
import pandas as pd
import numpy as np

import time
import math
import os.path

# from tqdm import tnrange, notebook
from tqdm.notebook import tqdm

from datetime import timedelta, datetime
from dateutil import parser

# Import the plotting library
import matplotlib.pyplot as plt
# %matplotlib inline

import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
from matplotlib.dates import MonthLocator

import seaborn as sns
sns.set()

plt.rcParams.update({'figure.figsize':(15,7), 'figure.dpi':120})
# plt.style.use('ggplot')

ModuleNotFoundError: No module named 'matplotlib'

## Load Data

In [ ]:
crypto = pd.read_csv('../outputs/btcusd_bars.csv', parse_dates=['time'])

def cleanPx(prices, freq='15T'):
    prices = prices.iloc[prices.time.drop_duplicates(keep='last').index]
    prices.time = pd.to_datetime(prices.time)
    prices.set_index('time', inplace=True)

    prices_ohlc = prices[['open','high','low','close']]
    prices_vol = prices[['volume']]

    prices_ohlc = prices_ohlc.resample(freq).agg({'open': 'first', 
                                 'high': 'max', 
                                 'low': 'min', 
                                 'close': 'last'})
    prices_vol = prices_vol.resample(freq).sum()

    prices = pd.concat([prices_ohlc, prices_vol], axis=1)
    prices.index = prices.index.tz_localize('UTC').tz_convert('Asia/Singapore')

    return prices.dropna()

crypto = cleanPx(crypto, '15T')
crypto.reset_index(inplace=True)
crypto.columns = ['Time', 'Open', 'High', 'Low', 'Close', 'Volume']
crypto.set_index('Time', inplace=True)

crypto

## Patterns for Detection

In [ ]:
import talib
candle_names = talib.get_function_groups()['Pattern Recognition']
removed = ['CDLCOUNTERATTACK', 'CDLLONGLINE', 'CDLSHORTLINE', 
           'CDLSTALLEDPATTERN', 'CDLKICKINGBYLENGTH']
candle_names = [name for name in candle_names if name not in removed]

In [ ]:
# interested = ['CDLDOJI', 'CDLDOJISTAR','CDLENGULFING']
# candle_names = [name for name in candle_names if name in interested]

# len(candle_names)

In [ ]:
', '.join(candle_names)

In [ ]:
crypto.reset_index(inplace=True)
crypto = crypto[['Date', 'Open', 'High', 'Low', 'Close']]
crypto.columns = ['time', 'open', 'high', 'low', 'close']

## Ranking of Patterns
Candlestick pattern ranking so that the most important candlestick pattern is selected for the candle.

In [ ]:
candle_rankings = {
        "CDL3LINESTRIKE_Bull": 1,
        "CDL3LINESTRIKE_Bear": 2,
        "CDL3BLACKCROWS_Bull": 3,
        "CDL3BLACKCROWS_Bear": 3,
        "CDLEVENINGSTAR_Bull": 4,
        "CDLEVENINGSTAR_Bear": 4,
        "CDLTASUKIGAP_Bull": 5,
        "CDLTASUKIGAP_Bear": 5,
        "CDLINVERTEDHAMMER_Bull": 6,
        "CDLINVERTEDHAMMER_Bear": 6,
        "CDLMATCHINGLOW_Bull": 7,
        "CDLMATCHINGLOW_Bear": 7,
        "CDLABANDONEDBABY_Bull": 8,
        "CDLABANDONEDBABY_Bear": 8,
        "CDLBREAKAWAY_Bull": 10,
        "CDLBREAKAWAY_Bear": 10,
        "CDLMORNINGSTAR_Bull": 12,
        "CDLMORNINGSTAR_Bear": 12,
        "CDLPIERCING_Bull": 13,
        "CDLPIERCING_Bear": 13,
        "CDLSTICKSANDWICH_Bull": 14,
        "CDLSTICKSANDWICH_Bear": 14,
        "CDLTHRUSTING_Bull": 15,
        "CDLTHRUSTING_Bear": 15,
        "CDLINNECK_Bull": 17,
        "CDLINNECK_Bear": 17,
        "CDL3INSIDE_Bull": 20,
        "CDL3INSIDE_Bear": 56,
        "CDLHOMINGPIGEON_Bull": 21,
        "CDLHOMINGPIGEON_Bear": 21,
        "CDLDARKCLOUDCOVER_Bull": 22,
        "CDLDARKCLOUDCOVER_Bear": 22,
        "CDLIDENTICAL3CROWS_Bull": 24,
        "CDLIDENTICAL3CROWS_Bear": 24,
        "CDLMORNINGDOJISTAR_Bull": 25,
        "CDLMORNINGDOJISTAR_Bear": 25,
        "CDLXSIDEGAP3METHODS_Bull": 27,
        "CDLXSIDEGAP3METHODS_Bear": 26,
        "CDLTRISTAR_Bull": 28,
        "CDLTRISTAR_Bear": 76,
        "CDLGAPSIDESIDEWHITE_Bull": 46,
        "CDLGAPSIDESIDEWHITE_Bear": 29,
        "CDLEVENINGDOJISTAR_Bull": 30,
        "CDLEVENINGDOJISTAR_Bear": 30,
        "CDL3WHITESOLDIERS_Bull": 32,
        "CDL3WHITESOLDIERS_Bear": 32,
        "CDLONNECK_Bull": 33,
        "CDLONNECK_Bear": 33,
        "CDL3OUTSIDE_Bull": 34,
        "CDL3OUTSIDE_Bear": 39,
        "CDLRICKSHAWMAN_Bull": 35,
        "CDLRICKSHAWMAN_Bear": 35,
        "CDLSEPARATINGLINES_Bull": 36,
        "CDLSEPARATINGLINES_Bear": 40,
        "CDLLONGLEGGEDDOJI_Bull": 37,
        "CDLLONGLEGGEDDOJI_Bear": 37,
        "CDLHARAMI_Bull": 38,
        "CDLHARAMI_Bear": 72,
        "CDLLADDERBOTTOM_Bull": 41,
        "CDLLADDERBOTTOM_Bear": 41,
        "CDLCLOSINGMARUBOZU_Bull": 70,
        "CDLCLOSINGMARUBOZU_Bear": 43,
        "CDLTAKURI_Bull": 47,
        "CDLTAKURI_Bear": 47,
        "CDLDOJISTAR_Bull": 49,
        "CDLDOJISTAR_Bear": 51,
        "CDLHARAMICROSS_Bull": 50,
        "CDLHARAMICROSS_Bear": 80,
        "CDLADVANCEBLOCK_Bull": 54,
        "CDLADVANCEBLOCK_Bear": 54,
        "CDLSHOOTINGSTAR_Bull": 55,
        "CDLSHOOTINGSTAR_Bear": 55,
        "CDLMARUBOZU_Bull": 71,
        "CDLMARUBOZU_Bear": 57,
        "CDLUNIQUE3RIVER_Bull": 60,
        "CDLUNIQUE3RIVER_Bear": 60,
        "CDL2CROWS_Bull": 61,
        "CDL2CROWS_Bear": 61,
        "CDLBELTHOLD_Bull": 62,
        "CDLBELTHOLD_Bear": 63,
        "CDLHAMMER_Bull": 65,
        "CDLHAMMER_Bear": 65,
        "CDLHIGHWAVE_Bull": 67,
        "CDLHIGHWAVE_Bear": 67,
        "CDLSPINNINGTOP_Bull": 69,
        "CDLSPINNINGTOP_Bear": 73,
        "CDLUPSIDEGAP2CROWS_Bull": 74,
        "CDLUPSIDEGAP2CROWS_Bear": 74,
        "CDLGRAVESTONEDOJI_Bull": 77,
        "CDLGRAVESTONEDOJI_Bear": 77,
        "CDLHIKKAKEMOD_Bull": 82,
        "CDLHIKKAKEMOD_Bear": 81,
        "CDLHIKKAKE_Bull": 85,
        "CDLHIKKAKE_Bear": 83,
        "CDLENGULFING_Bull": 84,
        "CDLENGULFING_Bear": 91,
        "CDLMATHOLD_Bull": 86,
        "CDLMATHOLD_Bear": 86,
        "CDLHANGINGMAN_Bull": 87,
        "CDLHANGINGMAN_Bear": 87,
        "CDLRISEFALL3METHODS_Bull": 94,
        "CDLRISEFALL3METHODS_Bear": 89,
        "CDLKICKING_Bull": 96,
        "CDLKICKING_Bear": 102,
        "CDLDRAGONFLYDOJI_Bull": 98,
        "CDLDRAGONFLYDOJI_Bear": 98,
        "CDLCONCEALBABYSWALL_Bull": 101,
        "CDLCONCEALBABYSWALL_Bear": 101,
        "CDL3STARSINSOUTH_Bull": 103,
        "CDL3STARSINSOUTH_Bear": 103,
        "CDLDOJI_Bull": 104,
        "CDLDOJI_Bear": 104
    }

## Detection of Patterns

In [ ]:
# extract OHLC 
op = crypto['open']
hi = crypto['high']
lo = crypto['low']
cl = crypto['close']

# create columns for each pattern
for candle in candle_names:
    # below is same as;
    # df["CDL3LINESTRIKE"] = talib.CDL3LINESTRIKE(op, hi, lo, cl)
    crypto[candle] = getattr(talib, candle)(op, hi, lo, cl)

## Naming of Patterns

In [ ]:
from itertools import compress

crypto['candlestick_pattern'] = np.nan
crypto['candlestick_match_count'] = np.nan

for index, row in crypto.iterrows():

    # no pattern found
    if len(row[candle_names]) - sum(row[candle_names] == 0) == 0:
        crypto.loc[index,'candlestick_pattern'] = "NO_PATTERN"
        crypto.loc[index, 'candlestick_match_count'] = 0
    # single pattern found
    elif len(row[candle_names]) - sum(row[candle_names] == 0) == 1:
        # bull pattern 100 or 200
        if any(row[candle_names].values > 0):
            pattern = list(compress(row[candle_names].keys(), row[candle_names].values != 0))[0] + '_Bull'
            crypto.loc[index, 'candlestick_pattern'] = pattern
            crypto.loc[index, 'candlestick_match_count'] = 1
        # bear pattern -100 or -200
        else:
            pattern = list(compress(row[candle_names].keys(), row[candle_names].values != 0))[0] + '_Bear'
            crypto.loc[index, 'candlestick_pattern'] = pattern
            crypto.loc[index, 'candlestick_match_count'] = 1
    # multiple patterns matched -- select best performance
    else:
        # filter out pattern names from bool list of values
        patterns = list(compress(row[candle_names].keys(), row[candle_names].values != 0))
        container = []
        for pattern in patterns:
            if row[pattern] > 0:
                container.append(pattern + '_Bull')
            else:
                container.append(pattern + '_Bear')
        rank_list = [candle_rankings[p] for p in container]
        if len(rank_list) == len(container):
            rank_index_best = rank_list.index(min(rank_list))
            crypto.loc[index, 'candlestick_pattern'] = container[rank_index_best]
            crypto.loc[index, 'candlestick_match_count'] = len(container)


In [ ]:
# clean up candle columns
try:
    crypto.drop(candle_names, axis = 1, inplace = True)
except:
    pass

crypto.loc[crypto.candlestick_pattern == 'NO_PATTERN', 'candlestick_pattern'] = ''
crypto.candlestick_pattern = crypto.candlestick_pattern.apply(lambda x: x[3:])

## Saving the output

In [ ]:
OUTPUT_FOLDER = '../output/'
crypto.to_csv(OUTPUT_FOLDER + 'ethusd.csv', index=False)

## Visualisation
See the visualisation on Tableau here: [Candlestick Patterns](https://public.tableau.com/profile/edsicage#!/vizhome/MachineTrading/CandlestickPattern)

---
# END

## MSE Pattern Classification — Momentum · Reversal · Indecision

Every TA-Lib candlestick pattern classified for the **Model Selection Engine (MSE)**.

| Category | MSE Role | Model Bias |
|----------|----------|------------|
| **Momentum** | Strong directional conviction — enter at close or breakout | Model A / B |
| **Reversal** | Potential turning point — wait for retrace confirmation | Model C |
| **Indecision** | No clear bias — lowest MSE confidence | Default B |
| **Continuation** | Trend continuation after pause — breakout bias | Model B |

In [ ]:
# ─── Full MSE Pattern Classification ────────────────────────────────────────
# Every TA-Lib Pattern Recognition function classified into exactly one category.
# This drives the MSE: momentum count > reversal → Model A; reversal ≥ momentum → Model C; else Model B.

import talib

ALL_PATTERNS = sorted(talib.get_function_groups()["Pattern Recognition"])

# ── MOMENTUM: Strong directional candles implying immediate continuation ──
MOMENTUM_PATTERNS = {
    "CDLMARUBOZU",            # Full-body candle, no wicks — pure momentum
    "CDLCLOSINGMARUBOZU",     # Full-body close — strong momentum variant
    "CDL3WHITESOLDIERS",      # Three consecutive bullish bodies — strong upside
    "CDL3BLACKCROWS",         # Three consecutive bearish bodies — strong downside
    "CDL3LINESTRIKE",         # Three-line strike (highest ranked in notebook)
    "CDLKICKING",             # Gap + marubozu — explosive directional move
    "CDLKICKINGBYLENGTH",     # Kicking by length variant
    "CDLBELTHOLD",            # Belt-hold: opens at extreme, strong close
    "CDLSEPARATINGLINES",     # Opens at previous close, moves strongly directional
    "CDLLONGLINE",            # Long body candle — directional strength
    "CDLIDENTICAL3CROWS",     # Three identical bearish candles — strong bearish momentum
    "CDL3OUTSIDE",            # Outside day pattern — breakout momentum
    "CDLMATHOLD",             # Mat-hold: continuation after brief consolidation
    "CDLRISEFALL3METHODS",    # Rising/falling three methods — trend continuation
    "CDLGAPSIDESIDEWHITE",    # Gap side-by-side white — gap continuation momentum
    "CDLTASUKIGAP",           # Tasuki gap — continuation with gap
}

# ── REVERSAL: Potential turning-point signals requiring confirmation ──
REVERSAL_PATTERNS = {
    "CDLDOJI",                # Classic doji — indeterminate but signals exhaustion
    "CDLDOJISTAR",            # Doji star — reversal after gap
    "CDLHAMMER",              # Hammer — bullish reversal at bottom
    "CDLINVERTEDHAMMER",      # Inverted hammer — bullish reversal candidate
    "CDLHANGINGMAN",          # Hanging man — bearish reversal at top
    "CDLSHOOTINGSTAR",        # Shooting star — bearish reversal
    "CDLENGULFING",           # Engulfing — strong reversal signal
    "CDLHARAMI",              # Harami — inside bar reversal
    "CDLHARAMICROSS",         # Harami cross — doji inside bar
    "CDLTAKURI",              # Takuri (dragonfly doji at bottom)
    "CDLDRAGONFLYDOJI",       # Dragonfly doji — bullish reversal
    "CDLGRAVESTONEDOJI",      # Gravestone doji — bearish reversal
    "CDLMORNINGSTAR",         # Morning star — 3-bar bullish reversal
    "CDLMORNINGDOJISTAR",     # Morning doji star variant
    "CDLEVENINGSTAR",         # Evening star — 3-bar bearish reversal
    "CDLEVENINGDOJISTAR",     # Evening doji star variant
    "CDLABANDONEDBABY",       # Abandoned baby — rare, strong reversal
    "CDLPIERCING",            # Piercing line — bullish reversal
    "CDLDARKCLOUDCOVER",      # Dark cloud cover — bearish reversal
    "CDLHOMINGPIGEON",        # Homing pigeon — bullish reversal (inside bar variant)
    "CDLMATCHINGLOW",         # Matching low — bullish reversal (double bottom)
    "CDLCOUNTERATTACK",       # Counter-attack — reversal at extremes
    "CDLBREAKAWAY",           # Breakaway — reversal after extended move
    "CDL3INSIDE",             # 3 inside — harami confirmation reversal
    "CDL3STARSINSOUTH",       # 3 stars in south — bullish reversal
    "CDLCONCEALBABYSWALL",    # Concealing baby swallow — bullish reversal
    "CDLLADDERBOTTOM",        # Ladder bottom — bullish reversal
    "CDLUNIQUE3RIVER",        # Unique 3 river bottom — bullish reversal
    "CDL2CROWS",              # 2 crows — bearish reversal (gap up, bear engulf)
    "CDLUPSIDEGAP2CROWS",     # Upside gap 2 crows — bearish reversal
    "CDLADVANCEBLOCK",        # Advance block — weakening uptrend = reversal warning
    "CDLSTALLEDPATTERN",      # Stalled pattern — uptrend exhaustion
    "CDLHIKKAKE",             # Hikkake — inside bar false-breakout reversal
    "CDLHIKKAKEMOD",          # Hikkake modified variant
    "CDLTRISTAR",             # Tri-star — three dojis = strong reversal
    "CDLSTICKSANDWICH",       # Stick sandwich — reversal
    "CDLINNECK",              # In-neck — weak continuation that often reverses
    "CDLONNECK",              # On-neck — similar to in-neck
    "CDLTHRUSTING",           # Thrusting — weak piercing = potential reversal
    "CDLXSIDEGAP3METHODS",    # X side gap 3 methods — gap fill reversal
}

# ── INDECISION: No clear directional bias ──
INDECISION_PATTERNS = {
    "CDLSPINNINGTOP",         # Spinning top — small body, long wicks
    "CDLHIGHWAVE",            # High wave — extreme wick length, body tiny
    "CDLRICKSHAWMAN",         # Rickshaw man — variant of high wave
    "CDLLONGLEGGEDDOJI",      # Long-legged doji — extended indecision
    "CDLSHORTLINE",           # Short line — small candle, low conviction
}

# ── Validation ──
all_classified = MOMENTUM_PATTERNS | REVERSAL_PATTERNS | INDECISION_PATTERNS
all_talib = set(ALL_PATTERNS)
unclassified = all_talib - all_classified
extra = all_classified - all_talib

print(f"TA-Lib patterns:   {len(all_talib)}")
print(f"Classified:        {len(all_classified)}")
print(f"  Momentum:        {len(MOMENTUM_PATTERNS)}")
print(f"  Reversal:        {len(REVERSAL_PATTERNS)}")
print(f"  Indecision:      {len(INDECISION_PATTERNS)}")
print(f"  Overlaps:        {len(all_classified) - len(MOMENTUM_PATTERNS) - len(REVERSAL_PATTERNS) - len(INDECISION_PATTERNS)}")
print(f"Unclassified:      {len(unclassified)}: {sorted(unclassified) if unclassified else 'None'}")
print(f"Extra (not in lib): {len(extra)}: {sorted(extra) if extra else 'None'}")
assert len(unclassified) == 0, f"UNCLASSIFIED: {sorted(unclassified)}"
assert len(MOMENTUM_PATTERNS & REVERSAL_PATTERNS) == 0, "Overlap momentum/reversal!"
assert len(MOMENTUM_PATTERNS & INDECISION_PATTERNS) == 0, "Overlap momentum/indecision!"
assert len(REVERSAL_PATTERNS & INDECISION_PATTERNS) == 0, "Overlap reversal/indecision!"
print("\n✅ All 61 patterns classified. No overlaps. No gaps.")

In [ ]:
# ─── Classification Summary Table ───────────────────────────────────────────
import pandas as pd

rows = []
for p in sorted(ALL_PATTERNS):
    if p in MOMENTUM_PATTERNS:
        cat = "Momentum"
        model = "A / B"
        color = "🟢"
    elif p in REVERSAL_PATTERNS:
        cat = "Reversal"
        model = "C"
        color = "🔴"
    elif p in INDECISION_PATTERNS:
        cat = "Indecision"
        model = "B (default)"
        color = "🟡"
    else:
        cat = "UNCLASSIFIED"
        model = "?"
        color = "⚠️"
    # Get ranking from candle_rankings if available
    bull_rank = candle_rankings.get(f"{p}_Bull", "-")
    bear_rank = candle_rankings.get(f"{p}_Bear", "-")
    rows.append({"Pattern": p.replace("CDL", ""), "TA-Lib": p, "Category": cat,
                 "MSE Model": model, "Bull Rank": bull_rank, "Bear Rank": bear_rank, "": color})

class_df = pd.DataFrame(rows)
class_df = class_df.sort_values(["Category", "Pattern"])

# Summary
print("=" * 70)
print("MSE Pattern Classification Summary")
print("=" * 70)
for cat in ["Momentum", "Reversal", "Indecision"]:
    subset = class_df[class_df["Category"] == cat]
    print(f"\n{cat} ({len(subset)} patterns):")
    for _, r in subset.iterrows():
        print(f"  {r['']} {r['Pattern']:30s} ranks: Bull={str(r['Bull Rank']):>3s}  Bear={str(r['Bear Rank']):>3s}  → {r['MSE Model']}")

print(f"\nTotal: {len(class_df)} patterns classified")
class_df[["", "Pattern", "Category", "MSE Model", "Bull Rank", "Bear Rank"]]